# Recurring Donor Conversion Predictive

## Problem Framing

**Business question:** Which one-time donors are most likely to become recurring donors?

**Primary users:** fundraisers, development leadership

**Decision supported:** Document the recurring-conversion concept and current data gap without forcing a weak predictive model.

**Modeling goal:** Prediction. This notebook is judged first on how well it scores new, unseen records rather than on whether a single coefficient is easy to interpret.

**Target / outcome anchor:** Current data issue: recurring donations begin on first donation for the few recurring supporters, so a future conversion label is effectively absent in the available data.

**Submission status:** Supplemental predictive candidate. The notebook frames a meaningful business problem, but the repo does not currently commit a trained predictive artifact for this track.

Prediction is the right framing here because the organization needs an operational ranking or forecast that can be applied to future records. This notebook therefore prioritizes leakage control, reproducible feature availability at scoring time, and honest out-of-sample validation. Causal interpretation is handled more carefully in the paired explanatory notebook for the same pipeline.


## Data Acquisition, Preparation & Exploration

**Data source and grain:** This notebook loads `supporter_monthly_features`, which contains supporter-month snapshots that preserve timing signals needed for forward-looking donor prediction and outreach sequencing.

**Reproducible preparation pipeline:** The code cells below use the shared notebook bootstrap plus `load_notebook_context(...)` so the data loading, joins, cleaning, and feature engineering come from reusable modules under `ml/src/data/`, `ml/src/features/`, and the pipeline-specific implementation for `recurring_donor_conversion` rather than from one-off notebook code.

**Exploration focus:** Before trusting model output, review the shared table preview, target availability, missingness patterns, date coverage, outliers, and any fields that appear to encode post-outcome information. The goal is to confirm that the data can answer the business question without leakage.

**Shared references used by this notebook:**
* `ml/docs/data-joins.md`
* `ml/docs/feature-catalog.md`
* `ml/docs/phase-b-notebook-standardization.md`
* `ml/docs/phase-3-predictive-pipelines.md`
* `ml/docs/phase-4-modeling-framework.md`


In [1]:
print("Notebook execution proof: recurring_donor_conversion predictive template rendered successfully.")


Notebook execution proof: recurring_donor_conversion predictive template rendered successfully.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "ml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


In [ ]:
from ml.src.pipelines.notebook_support import (
    load_evaluation_bundle,
    load_notebook_context,
    summarize_frame,
)

context = load_notebook_context(
    pipeline_name='recurring_donor_conversion',
    dataset_name='supporter_monthly_features',
    predictive_impl=None,
    use_predictive_dataset=True,
)
pipeline_name = context["pipeline_name"]
dataset_name = context["dataset_name"]
config = context["config"]
dataset = context["dataset"]

summarize_frame(dataset)


## Modeling & Feature Selection

This notebook is prediction-first, so candidate models should be compared using the saved evaluation bundle and the strongest out-of-sample metric for this target. The preferred model is whichever generalizes best while still being stable enough to deploy in staff workflows.

Feature selection follows four rules in this submission:

1. Keep only fields available at scoring time.
2. Remove identifiers, direct future labels, and post-outcome leakage.
3. Favor features with stable business meaning and tolerable missingness.
4. Use domain reasoning alongside model importance so the final feature set is defensible.

**Committed artifact summary:** No committed predictive metrics are stored for this track in `ml/models`, so this notebook should be read as a scoped analytical submission aid rather than as proof of a finalized production score.

**Feature inventory:** No committed feature list engineered inputs are currently stored in `ml/models/recurring_donor_conversion/feature_list.json`.


## Evaluation & Interpretation

**Validation discipline:** The code cells below pull the saved metrics JSON, holdout comparison table, and cross-validation summary for this pipeline. Those artifacts are the correct basis for judging model quality, not in-sample fit.

**Business meaning of the score:** A high score means the record looks more likely to match this pipeline's target outcome on unseen future data, so `Document the recurring-conversion concept and current data gap without forcing a weak predictive model.` can be prioritized earlier.

**Real-world error consequences:** False negatives matter because the organization can miss donor retention, upgrade, or revenue opportunities. False positives matter because they can waste outreach effort, over-contact supporters, or shift resources toward weak campaigns.

Operationally, the right threshold depends on the workflow. Teams should read precision, recall, ranking quality, and class balance together before deciding whether the score should drive alerts, queue ordering, or a softer recommendation panel.


In [ ]:
evaluation = load_evaluation_bundle('recurring_donor_conversion')
metrics = evaluation["metrics"]
holdout_comparison = evaluation["holdout_comparison"]
cv_summary = evaluation["cv_summary"]

metrics


In [ ]:
summarize_frame(holdout_comparison)


In [ ]:
summarize_frame(cv_summary)


## Causal and Relationship Analysis

This notebook uses predictive relationships, not causal identification. Strong features matter because they help separate future outcomes on unseen data, but they do **not** prove that changing the feature will change the outcome.

**Most impactful committed features:** Committed feature importance artifacts are not present for this track, so this notebook uses domain reasoning and the shared data profile to discuss likely drivers.

**Decision guidance:** Use these high-signal features to focus staff review, choose follow-up questions, and prioritize intervention or outreach. When leadership wants to argue that a driver should become policy, treatment, or strategy, they should pair this notebook with the explanatory companion and with domain judgment about omitted variables, timing, and selection bias.


## Deployment Notes

**Canonical widgets for the web app:**
* `insight_summary_card`
* `recommendation_panel`

**Submission deployment notes:**
* Keep this as a notebook-only research track until the source data captures real recurring-conversion transitions.
* Use the notebook to document the target gap and propose what additional event history would unlock the model.

**Artifact locations referenced by this notebook:**
* `ml/models/recurring_donor_conversion/metrics.json`
* `ml/models/recurring_donor_conversion/feature_list.json`
* `ml/models/recurring_donor_conversion/explainability.csv`
* `ml/reports/evaluation/recurring_donor_conversion_metrics.json`

**Endpoint pattern used in the app layer when this track is scored:**
* `POST /ml/predict/recurring_donor_conversion`
* `POST /ml/score-batch/recurring_donor_conversion`
